# NuFrost Evaluation Notebook
This notebook runs the accuracy assessment for NuFrost, Zhu2015, and HANTS algorithms using simulated gap experiments.

## Configuration

In [ ]:
from pathlib import Path
from re import I

MOUNT_POINT_IN_COLAB = Path("/content/drive")
PROJECT_PATH_IN_GDRIVE = Path("WorkSpaces/nufrost")
PROJECT_DIR = MOUNT_POINT_IN_COLAB / "MyDrive" / PROJECT_PATH_IN_GDRIVE
IMAGE_DIR   = PROJECT_DIR / "data/input"
OUTPUT_DIR  = PROJECT_DIR / "data/output"
CACHE_DIR   = PROJECT_DIR / "data/cache"
OUTPUT_CSV_PATH = OUTPUT_DIR / "data/evaluation_results.csv"

IMAGE_NAMES = []

SAMPLE_POINTS_NUM = 50000

In [ ]:
import os
from google.colab import drive # type: ignore[import]

drive.mount(MOUNT_POINT_IN_COLAB.as_posix())
os.chdir(PROJECT_DIR)
print(f"[Working directory changed to: {os.getcwd()}]")

In [ ]:
%pip install -r requirements.txt

In [ ]:
import importlib
import src.evaluation
import pandas as pd

from config import build_args
from IPython.display import display

# Ensure the latest version of the module is loaded
importlib.reload(src.evaluation)

# To run specific images, put their names in this list.
# If the list is empty, it will automatically run ALL .tif images in the folder.

if IMAGE_NAMES:
    image_paths = [IMAGE_DIR / name for name in IMAGE_NAMES]
else:
    if IMAGE_DIR.exists() and IMAGE_DIR.is_dir():
        image_paths = [f for f in IMAGE_DIR.iterdir() if f.is_file() and f.suffix == '.tif']
    else:
        print(f"Folder not found: {IMAGE_DIR}")
        raise FileNotFoundError(f"Folder not found: {IMAGE_DIR}")

print(f"Found {len(image_paths)} images to evaluate.")


In [ ]:
print("========== Starting Accuracy Assessment ==========")

all_results = []

for image_path in image_paths:
    print(f"\n--- Evaluating: {image_path.name} ---")
    if not image_path.exists():
        print(f"[Warning] File not found: {image_path}. Skipping.")
        continue

    # 1. Initialize arguments
    args = build_args({})
    args.image = image_path.as_posix()  # Ensure the image path is set for Colab
    args.n_jobs = -1  # Set -1 to use all available cores in Colab
    args.cache_dir = CACHE_DIR.as_posix()  # Ensure cache directory is set for Colab

    # 2. Run evaluation
    df_results = src.evaluation.evaluate_algorithms(
        image_path=args.image,
        args=args,
        num_points=SAMPLE_POINTS_NUM
    )

    # Add image name column to distinguish results
    df_results.insert(0, "Image", image_name.)
    all_results.append(df_results)

# 3. Combine and save results
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)

    # Ensure output directory exists
    Path(OUTPUT_CSV_PATH).parent.mkdir(parents=True, exist_ok=True)

    # Save to CSV
    final_df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"\n[Success] All results saved to: {OUTPUT_CSV_PATH}")

    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    display(final_df)
else:
    print("No valid images processed.")
